# Raw CSV Exploration — CSIC 2010

## Goal

Understand the structure of the raw CSIC 2010 HTTP CSV and derive a precise specification for the `preprocess` stage.

## Questions to answer

1. What does one raw row represent?
2. Is `index` a globally unique request identifier or only locally meaningful?
3. Which columns are constant within a logical request?
4. Which columns contain missing values, and what do those missing values mean?
5. How should multiple raw rows be aggregated into one request-level row?
6. What fields should the preprocess stage preserve, normalize, or add?

## Expected output of this notebook

By the end of this notebook, we should be able to define the preprocess-stage contract:

- input CSV path and schema
- grouping key strategy
- aggregation rules per column
- missing-value policy
- output CSV schema
- validation checks
# output_http_csic_2010_weka_with_duplications_RAW-RFC2616_escd_v02_full.csv

In [1]:
import os
import argparse
from pathlib import Path

import pandas as pd

In [2]:
basepath = Path(os.environ["PWD"]) / "mlhttp"
RAW_CSV = Path(f"{basepath}/data/raw/output_http_csic_2010_weka_with_duplications_RAW-RFC2616_escd_v02_full.csv")

df_raw = pd.read_csv(
    RAW_CSV,
    dtype={"contentType": "string"},
    low_memory=False,
)

print(f"Raw CSV path: {RAW_CSV}")
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

Raw CSV path: /home/jovyan/my-work/mlhttp/data/raw/output_http_csic_2010_weka_with_duplications_RAW-RFC2616_escd_v02_full.csv
Raw shape: (223585, 18)


,index,method,url,protocol,userAgent,pragma,cacheControl,accept,acceptEncoding,acceptCharset,acceptLanguage,host,connection,contentLength,contentType,cookie,payload,label
0,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,<NA>,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,id=2,anom
1,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,<NA>,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,nombre=Jam%F3n+Ib%E9rico,anom
2,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,<NA>,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,precio=85,anom
3,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,<NA>,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+...,anom
4,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,<NA>,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,B1=A%F1adir+al+carrito,anom


The raw CSV contains 223,585 rows and 18 columns.
The first few rows have `index = 0`. 
The `payload` column contains different payload fragments in `key=value` format.
- `id=2`
- `nombre=Jam%F3n+Ib%E9rico`
- `precio=85`
- `cantidad=...`
- `B1=...`

Therefore:

- `index:` It is not necessarily globally unique. 
- `payload` fragments requires aggregation before modeling. 

So an HTTP packet may be partitioned into multiple rows identified by same index. 

In [68]:
# First few lines of the file
df_raw[["index", "label"]].head(10)

,index,label
0,0,anom
1,0,anom
2,0,anom
3,0,anom
4,0,anom
5,1,anom
6,1,anom
7,1,anom
8,1,anom
9,1,anom


File begins with label `anom`

In [67]:
# last few lines of the file
df_raw[["index", "label"]].tail(10)

,index,label
223575,35999,norm
223576,35999,norm
223577,35999,norm
223578,35999,norm
223579,35999,norm
223580,35999,norm
223581,35999,norm
223582,35999,norm
223583,35999,norm
223584,35999,norm


File ends with label `norm`

Count `anom` & `norm`

df_raw["label"].value_counts()

In [70]:
# inspect the boundary
df_raw.loc[119580:119590, ["index", "label"]]

,index,label
119580,25064,anom
119581,25064,anom
119582,25064,anom
119583,25064,anom
119584,25064,anom
119585,0,norm
119586,1,norm
119587,1,norm
119588,1,norm
119589,1,norm


This shows the rows where `label` changes and `index` restarts to `0`  
- At row `119584`, `index` is `25064` and label is `anom`  
- The next `119585`, `index` `0` and label `norm`  

`index` alone is not globally unique. `index = 0` appears in both the anomalous and normal

The logically there are two partitions  
- `anom` for rows before the transition
- `norm` for rows at and after the transition

Unique request identifier can be derived as:  
    `request_id = f"{partition}_{index}"`

The preprocess stage must create a source-aware request identifier:

    source_partition = "anom" or "norm"
    request_id = source_partition + "_" + index

Summary:  
Following transformation are required on raw parameter-level CSV to convert it into one-row-per-request CSV  
- create globally unique request_id from source partition and index  
- aggregate payload fragments  
- retain and validate request metadata  
- create missingness indicators  
- fill selected missing values  